# Phase 2.5: Data Quality & Integrity Audit
## Open University Learning Analytics Dataset (OULAD)

**Objective**: Perform an empirical, non-destructive audit of missing values, sentinel strings (`'?'`), exact duplicates, primary/foreign key integrity, and numerical/categorical constraints across all seven raw OULAD tables.

In [ ]:
import sys
sys.path.append('..')
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_quality_audit import OULADQualityAuditor

RAW_DIR = '../data/raw'
print('Libraries imported successfully.')

### 1. Execute Modular Audit via `OULADQualityAuditor`

In [ ]:
auditor = OULADQualityAuditor(raw_data_dir=RAW_DIR)
results = auditor.run_full_audit()
print('Audit completed across all 7 tables.')

### 2. Missing & Sentinel (`'?'`) Value Summary Table

In [ ]:
missing_rows = []
for table, data in results.items():
    total_rows = data['total_rows']
    for col, q_cnt in data['sentinel_q_values'].items():
        null_cnt = data['missing_values'][col]
        if q_cnt > 0 or null_cnt > 0:
            missing_rows.append({
                'Table': table,
                'Column': col,
                'Total Rows': total_rows,
                'Sentinel (?) Count': q_cnt,
                'Standard Null Count': null_cnt,
                'Total Missing (%)': round((q_cnt + null_cnt) / total_rows * 100, 3)
            })

df_missing = pd.DataFrame(missing_rows)
display(df_missing)

### 3. Duplicate Analysis Across All Tables

Notice the key distinction in `studentVle.csv`:
- Exact duplicate rows (identical clicks across all 6 columns) = 787,170 rows (7.39%).
- Repeated `(student, site, date)` interaction events across multi-session activity = 2,195,949 records.

In [ ]:
dup_summary = []
for table, data in results.items():
    dup_summary.append({
        'Table': table,
        'Total Rows': data['total_rows'],
        'Exact Duplicate Rows': data['exact_duplicate_rows'],
        'Key Duplicates': data.get('key_duplicates') or data.get('composite_key_duplicates') or data.get('repeated_key_combinations', 0)
    })

df_dups = pd.DataFrame(dup_summary)
display(df_dups)

### 4. Investigation of Key Column Anomalies

#### 4.1 `studentInfo.imd_band` Missingness by Geographic Region
89.7% of all missing `imd_band` entries originate from the North Region and Ireland, explained by administrative postal code mapping constraints between national deprivation decile indices.

In [ ]:
si = pd.read_csv(f'{RAW_DIR}/studentInfo.csv')
missing_imd = si[si['imd_band'] == '?']['region'].value_counts()
print('Missing IMD Band by Region:')
print(missing_imd)

#### 4.2 Negative Submission Dates in `studentAssessment.csv`
- 1,909 rows are credit transfers from prior attempts (`is_banked == 1`), systematically logged with negative dates.
- 148 rows are legitimate early submissions prior to day 0 (`is_banked == 0`).

In [ ]:
sa = pd.read_csv(f'{RAW_DIR}/studentAssessment.csv')
neg_submissions = sa[sa['date_submitted'] < 0]
print('Breakdown of Negative Submission Dates by is_banked status:')
print(neg_submissions['is_banked'].value_counts())

### 5. Audit Conclusions & Phase 3 Recommendations

1. **Sentinel Ingestion**: Coerce `'?'` -> `np.nan` on load across all tables.
2. **Unregistration vs Final Result**: `date_unregistration` nulls are structurally expected (students who did not withdraw).
3. **Exam Grade Asymmetry**: Exam scores only exist in `studentAssessment.csv` for modules CCC and DDD. Early-warning models must rely on continuous coursework (TMAs/CMAs) and clickstream interactions.
4. **Clickstream Aggregations**: Sum `sum_click` per `(code_module, code_presentation, id_student, date)` in Phase 5 to cleanly combine session logs.